# KPCA + LDA for Multi-Class Classification

"This notebook implements Kernel Principal Component Analysis (KPCA) followed by Linear Discriminant Analysis (LDA) for multi-class classification problems. The implementation will be tested on CIFAR-10."

In [1]:
# !pip install numpy
# !pip install scikit-learn
# !pip install torch
# !pip install pandas
# !pip install matplotlib
# !pip install torchvision
# !pip install seaborn

In [2]:
import random
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import eigh
import sklearn.metrics
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [3]:
# Reproducibility, utilities and config
import os, time, random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Config
SUBSET_SIZE = 2000 
RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Helper functions
def squared_dist(X, Y):
    X2 = X.reshape(X.shape[0], -1)
    Y2 = Y.reshape(Y.shape[0], -1)
    XX = (X2**2).sum(axis=1)[:, None]
    YY = (Y2**2).sum(axis=1)[None, :]
    return XX + YY - 2 * (X2 @ Y2.T)

def rbf_kernel(X, Y, gamma):
    D = squared_dist(X, Y)
    return np.exp(-gamma * D)

print('Config: SUBSET_SIZE=', SUBSET_SIZE, 'SEED=', SEED)


Config: SUBSET_SIZE= 2000 SEED= 42


## Data Loading and Preprocessing

Load CIFAR-10 (or use local files if present). Apply standard normalization and flatten images to 1D vectors before kernel computation. Use a smaller subset for KPCA to reduce memory and runtime during experiments.

In [4]:
import os

mean = (0.49139968, 0.48215827 ,0.44653124)
std = (0.24703233, 0.24348505, 0.26158768)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    torch.flatten
])

train_size = 50000
test_size = 10000

# Use existing CIFAR-10 files if present; otherwise attempt to download
cifar_dir = os.path.join('./data', 'cifar-10-batches-py')
if os.path.isdir(cifar_dir):
    download_flag = False
    print('Found local CIFAR-10 at', cifar_dir, '- not downloading')
else:
    download_flag = True
    print('CIFAR-10 not found locally; attempting to download (this may take a while)')

try:
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=download_flag, transform=transform)
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=download_flag, transform=transform)
except Exception as e:
    # Informative error and fallback advice
    print('Dataset download or load failed:', e)
    print('If you have the data files, place them in ./data/cifar-10-batches-py and re-run the cell;')
    print('Alternatively retry later with a working network connection.')
    raise

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, shuffle=True, batch_size=train_size, num_workers=2)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, shuffle=True, batch_size=test_size, num_workers=2)

images_train, labels_train = next(iter(train_loader))
images_test, labels_test = next(iter(test_loader))

images_train = images_train.numpy()
labels_train = labels_train.numpy()
images_test = images_test.numpy()
labels_test = labels_test.numpy()

classes = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print("CIFAR-10 loaded:")
print(f"Train shape: {images_train.shape}, labels: {labels_train.shape}")
print(f"Test shape: {images_test.shape}, labels: {labels_test.shape}")

Found local CIFAR-10 at ./data\cifar-10-batches-py - not downloading
CIFAR-10 loaded:
Train shape: (50000, 3072), labels: (50000,)
Test shape: (10000, 3072), labels: (10000,)


## Kernel Principal Component Analysis (KPCA) Implementation

KPCA uses an RBF kernel to capture nonlinear structure. We compute the centered kernel matrix, perform eigen-decomposition, and keep the top components for projection of new samples. Keep subset sizes small to stay within memory limits.

In [5]:
class KPCA:
    """Vectorized and numerically stable KPCA (RBF kernel supported).

    Usage:
        kpca = KPCA(gamma=1e-5, n_components=100, tol=1e-12)
        X_train_k = kpca.fit_transform(X_train)
        X_test_k = kpca.transform_batch(X_test, batch_size=500)
    """
    def __init__(self, gamma=None, n_components=None, tol=1e-12):
        self.gamma = gamma
        self.n_components = n_components
        self.tol = tol
        self.alphas = None
        self.lambdas = None
        self.X_fit = None
        self.K_fit = None
        self.K_fit_row_mean = None
        self.K_fit_mean = None

    def _rbf(self, X, Y):
        X2 = X.reshape(X.shape[0], -1)
        Y2 = Y.reshape(Y.shape[0], -1)
        if self.gamma is None:
            self.gamma = 1.0 / X2.shape[1]
        D = (X2**2).sum(axis=1)[:, None] + (Y2**2).sum(axis=1)[None, :] - 2 * X2.dot(Y2.T)
        return np.exp(-self.gamma * D)

    def fit(self, X):
        self.X_fit = X
        K = self._rbf(X, X)
        self.K_fit = K

        # Center K: use row/col means for stable centering
        K_row_mean = K.mean(axis=0)
        K_col_mean = K.mean(axis=1)
        K_mean = K.mean()
        Kc = K - K_row_mean[None, :] - K_col_mean[:, None] + K_mean

        # Eigendecomposition (symmetric)
        lambdas, alphas = eigh(Kc)
        idx = np.argsort(lambdas)[::-1]
        lambdas = lambdas[idx]
        alphas = alphas[:, idx]

        # keep eigenvalues above tolerance
        positive = lambdas > self.tol
        lambdas = lambdas[positive]
        alphas = alphas[:, positive]

        if self.n_components is not None:
            lambdas = lambdas[:self.n_components]
            alphas = alphas[:, :self.n_components]

        # Normalize eigenvectors for projection (divide by sqrt(lambda))
        alphas = alphas / np.sqrt(lambdas[None, :])

        self.lambdas = lambdas
        self.alphas = alphas
        self.K_fit_row_mean = K_row_mean
        self.K_fit_mean = K_mean

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    def transform(self, X):
        if self.X_fit is None:
            raise ValueError('KPCA not fitted yet')
        K_test = self._rbf(X, self.X_fit)
        # center K_test using training quantities
        K_test_row_mean = K_test.mean(axis=1)
        Kc_test = K_test - self.K_fit_row_mean[None, :] - K_test_row_mean[:, None] + self.K_fit_mean
        X_transformed = Kc_test.dot(self.alphas)
        return X_transformed

    def transform_batch(self, X, batch_size=500):
        """Transform X in batches to reduce memory usage."""
        n = X.shape[0]
        parts = []
        for i in range(0, n, batch_size):
            Xb = X[i:i+batch_size]
            parts.append(self.transform(Xb))
        return np.vstack(parts)

## Linear Discriminant Analysis (LDA) Implementation

Short description:

LDA finds directions that maximize class separation (between-class variance / within-class variance). Here we train LDA on KPCA features to produce a classifier using nearest-class-mean decision in the projected space.

In [6]:
class LDAClassifier:
    """Simple LDA classifier that precomputes projected class means for fast prediction."""
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.W = None
        self.class_means = None
        self.classes = None
        self.class_means_proj = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_features = X.shape[1]

        # Compute class means
        self.class_means = {}
        for c in self.classes:
            self.class_means[c] = np.mean(X[y == c], axis=0)

        # Overall mean
        overall_mean = np.mean(X, axis=0)

        # Within-class scatter
        S_W = np.zeros((n_features, n_features))
        for c in self.classes:
            X_c = X[y == c]
            mean_c = self.class_means[c]
            S_W += (X_c - mean_c).T @ (X_c - mean_c)

        # Between-class scatter
        S_B = np.zeros((n_features, n_features))
        for c in self.classes:
            n_c = np.sum(y == c)
            mean_c = self.class_means[c]
            S_B += n_c * np.outer(mean_c - overall_mean, mean_c - overall_mean)

        # Solve generalized eigenvalue problem
        eigvals, eigvecs = eigh(S_B, S_W)
        idx = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx]

        if self.n_components is None:
            self.n_components = min(n_features, n_classes - 1)

        self.W = eigvecs[:, :self.n_components]

        # Precompute projected class means for fast predict
        self.class_means_proj = {c: (self.class_means[c] @ self.W) for c in self.classes}

    def transform(self, X):
        return X @ self.W

    def predict(self, X):
        X_proj = self.transform(X)
        preds = []
        class_list = list(self.classes)
        class_proj_arr = np.vstack([self.class_means_proj[c] for c in class_list])
        # fast nearest mean (Euclidean)
        for x in X_proj:
            dists = np.linalg.norm(class_proj_arr - x[None, :], axis=1)
            preds.append(class_list[np.argmin(dists)])
        return np.array(preds)

## Applying KPCA + LDA on CIFAR-10

In [7]:
# Use a subset for KPCA due to computational cost
subset_size = 2000
X_train_subset = images_train[:subset_size]
y_train_subset = labels_train[:subset_size]
X_test_subset = images_test[:1000]
y_test_subset = labels_test[:1000]

print(f"Using subset: Train {X_train_subset.shape}, Test {X_test_subset.shape}")

# Apply KPCA
kpca = KPCA(gamma=1e-5, n_components=100)  # Adjust gamma and n_components
kpca.fit(X_train_subset)
X_train_kpca = kpca.transform(X_train_subset)
X_test_kpca = kpca.transform(X_test_subset)

print(f"KPCA transformed: Train {X_train_kpca.shape}, Test {X_test_kpca.shape}")

# Apply LDA
lda = LDAClassifier(n_components=9)  # C-1 = 9 for 10 classes
lda.fit(X_train_kpca, y_train_subset)

# Predict
y_pred = lda.predict(X_test_kpca)

# Evaluate
accuracy = accuracy_score(y_test_subset, y_pred)
print(f"KPCA+LDA Accuracy on CIFAR-10 subset: {accuracy:.4f}")
print(classification_report(y_test_subset, y_pred, target_names=classes))

Using subset: Train (2000, 3072), Test (1000, 3072)
KPCA transformed: Train (2000, 100), Test (1000, 100)
KPCA+LDA Accuracy on CIFAR-10 subset: 0.3510
              precision    recall  f1-score   support

    airplane       0.49      0.45      0.47       106
  automobile       0.37      0.40      0.38       106
        bird       0.19      0.24      0.21        89
         cat       0.21      0.20      0.21        84
        deer       0.29      0.32      0.30        91
         dog       0.36      0.34      0.35       107
        frog       0.36      0.38      0.37       106
       horse       0.36      0.32      0.34        99
        ship       0.49      0.49      0.49       103
       truck       0.38      0.33      0.35       109

    accuracy                           0.35      1000
   macro avg       0.35      0.35      0.35      1000
weighted avg       0.36      0.35      0.35      1000



In [8]:
# Plot eigenvalue spectrum with cumulative explained and export vector/PNG
lambdas = kpca.lambdas
total = lambdas.sum() if lambdas.size>0 else 1.0
cum = np.cumsum(lambdas) / total
fig, ax1 = plt.subplots(figsize=(9,4), dpi=200)
ax1.plot(np.arange(1, len(lambdas)+1), lambdas, marker='o', label='Eigenvalues')
ax1.set_yscale('log')
ax1.set_xlabel('Component')
ax1.set_ylabel('Eigenvalue (log scale)')
ax2 = ax1.twinx()
ax2.plot(np.arange(1, len(lambdas)+1), cum, color='C1', marker='x', label='Cumulative explained')
ax2.set_ylabel('Cumulative explained')
ax1.grid(True)
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines+lines2, labels+labels2, loc='best')
plt.title('KPCA eigenvalue spectrum and cumulative explained')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kpca_eigen_spectrum.png'), dpi=300)
print('Saved kpca_eigen_spectrum.png')
plt.close()

# KPCA 2D scatter (train) - clearer markers, legend and vector export
fig, ax = plt.subplots(figsize=(7,6), dpi=200)
scatter = ax.scatter(X_train_kpca[:,0], X_train_kpca[:,1], c=y_train_subset, cmap='tab10', s=25, alpha=0.65, edgecolors='none')
ax.set_title('KPCA: first two components (train)')
ax.set_xlabel('KPCA1'); ax.set_ylabel('KPCA2')
handles, _ = scatter.legend_elements()
ax.legend(handles, classes, bbox_to_anchor=(1.05,1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kpca_2d_train.png'), dpi=300)
print('Saved kpca_2d_train.png')
plt.close()

# LDA 2D scatter (train) - clearer markers and legend
X_train_lda = lda.transform(X_train_kpca)
fig, ax = plt.subplots(figsize=(7,6), dpi=200)
scatter = ax.scatter(X_train_lda[:,0], X_train_lda[:,1], c=y_train_subset, cmap='tab10', s=28, alpha=0.7, edgecolors='none')
ax.set_title('LDA projection (train)')
ax.set_xlabel('LD1'); ax.set_ylabel('LD2')
handles, _ = scatter.legend_elements()
ax.legend(handles, classes, bbox_to_anchor=(1.05,1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'lda_2d_train.png'), dpi=300)
print('Saved lda_2d_train.png')
plt.close()

Saved kpca_eigen_spectrum.png
Saved kpca_2d_train.png
Saved lda_2d_train.png


"Saving plots for CIFAR-10: KPCA+LDA projections and confusion matrices"

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# Project test data with LDA
X_test_lda = lda.transform(X_test_kpca)

# Scatter (test) - separate file for clarity
fig, ax = plt.subplots(figsize=(8,6), dpi=200)
scatter = ax.scatter(X_test_lda[:,0], X_test_lda[:,1], c=y_test_subset, cmap='tab10', s=30, alpha=0.75, edgecolors='none')
ax.set_title('CIFAR-10 KPCA+LDA projection (test)')
ax.set_xlabel('LD1'); ax.set_ylabel('LD2')
handles, _ = scatter.legend_elements()
ax.legend(handles, classes, bbox_to_anchor=(1.05,1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cifar10_kpca_lda_scatter.png'), dpi=300)
print('Saved cifar10_kpca_lda_scatter.png')
plt.close()

# Confusion matrix - annotated and labeled
cm = confusion_matrix(y_test_subset, y_pred)
fig, ax = plt.subplots(figsize=(9,7), dpi=200)
sns.heatmap(cm, fmt='d', cmap='viridis', annot=True, ax=ax)
ax.set_title('Confusion matrix CIFAR-10')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_xticklabels(classes, rotation=45)
ax.set_yticklabels(classes, rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'cifar10_kpca_lda_confusion.png'), dpi=300)
print('Saved cifar10_kpca_lda_confusion.png')
plt.close()

# Per-class precision/recall/f1 - seaborn grouped bar for clarity
p, r, f, sup = precision_recall_fscore_support(y_test_subset, y_pred, average=None, zero_division=0)
df_prf = pd.DataFrame({'class': classes, 'precision': p, 'recall': r, 'f1': f})
df_melt = df_prf.melt(id_vars='class', var_name='metric', value_name='score')
plt.figure(figsize=(10,4), dpi=200)
sns.barplot(data=df_melt, x='class', y='score', hue='metric', palette='Set2')
plt.ylim(0,1)
plt.ylabel('Score')
plt.title('Per-class precision / recall / F1')
plt.xticks(rotation=45)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'cifar10_perclass_prf.png'), dpi=300)
print('Saved cifar10_perclass_prf.png')
plt.close()

Saved cifar10_kpca_lda_scatter.png
Saved cifar10_kpca_lda_confusion.png
Saved cifar10_perclass_prf.png


## Comparison with Other Classifiers

The cell will save the KPCA eigenvalue spectrum, 2-D KPCA and LDA projections for training and testing samples, confusion matrices, and per-class precision/recall/F1 charts. 

In [10]:
# Compare with kNN on KPCA features
knn_kpca = KNeighborsClassifier(n_neighbors=3)
knn_kpca.fit(X_train_kpca, y_train_subset)
y_pred_knn_kpca = knn_kpca.predict(X_test_kpca)
acc_knn_kpca = accuracy_score(y_test_subset, y_pred_knn_kpca)
print(f"kNN on KPCA features: {acc_knn_kpca:.4f}")

# Compare with PCA + LDA
pca = PCA(n_components=100)
X_train_pca = pca.fit_transform(X_train_subset)
X_test_pca = pca.transform(X_test_subset)

lda_pca = LDAClassifier(n_components=9)
lda_pca.fit(X_train_pca, y_train_subset)
y_pred_lda_pca = lda_pca.predict(X_test_pca)
acc_lda_pca = accuracy_score(y_test_subset, y_pred_lda_pca)
print(f"PCA + LDA: {acc_lda_pca:.4f}")

# kNN on PCA
knn_pca = KNeighborsClassifier(n_neighbors=3)
knn_pca.fit(X_train_pca, y_train_subset)
y_pred_knn_pca = knn_pca.predict(X_test_pca)
acc_knn_pca = accuracy_score(y_test_subset, y_pred_knn_pca)
print(f"kNN on PCA features: {acc_knn_pca:.4f}")

# SVM on PCA
svm_pca = svm.SVC(kernel='rbf')
svm_pca.fit(X_train_pca, y_train_subset)
y_pred_svm_pca = svm_pca.predict(X_test_pca)
acc_svm_pca = accuracy_score(y_test_subset, y_pred_svm_pca)
print(f"SVM on PCA features: {acc_svm_pca:.4f}")

kNN on KPCA features: 0.2710
PCA + LDA: 0.3340
kNN on PCA features: 0.2670
SVM on PCA features: 0.4050


## Enhanced Evaluation
In this cell the custom KPCA + LDA methods are evaluated with other models: training & inference time, accuracy, balanced accuracy, macro precision/recall/F1, normalized confusion matrix, evaluation plots with averages and bar plots.

In [11]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score, confusion_matrix
import pandas as pd

# Prepare models to evaluate (use KPCA or PCA features as appropriate)
models = {
    'KPCA + LDA (custom)': ('kpca', lda),
    'kNN (KPCA)': ('kpca', knn_kpca),
    'Logistic (KPCA)': ('kpca', LogisticRegression(max_iter=1000)),
    'RandomForest (KPCA)': ('kpca', RandomForestClassifier(n_estimators=100, n_jobs=-1)),
    'PCA + LDA (custom)': ('pca', lda_pca),
    'kNN (PCA)': ('pca', knn_pca),
    'SVM (PCA)': ('pca', svm_pca),
}

results = []

for name, (feat_type, model) in models.items():
    print(f"Evaluating: {name}")
    Xtr = X_train_kpca if feat_type == 'kpca' else X_train_pca
    Xte = X_test_kpca if feat_type == 'kpca' else X_test_pca

    # Fit (timed)
    t0 = time.time()
    model.fit(Xtr, y_train_subset)
    fit_time = time.time() - t0

    # Predict (timed)
    t1 = time.time()
    y_pred_m = model.predict(Xte)
    pred_time = time.time() - t1

    acc = accuracy_score(y_test_subset, y_pred_m)
    bal = balanced_accuracy_score(y_test_subset, y_pred_m)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test_subset, y_pred_m, average='macro', zero_division=0)

    cm = confusion_matrix(y_test_subset, y_pred_m)
    # handle rows with zero sum
    row_sums = cm.sum(axis=1)
    cm_norm = np.divide(cm.astype('float'), row_sums[:, None], out=np.zeros_like(cm, dtype=float), where=row_sums[:, None] != 0)

    results.append({
        'model': name,
        'accuracy': acc,
        'balanced_accuracy': bal,
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
        'fit_time_s': fit_time,
        'pred_time_s': pred_time,
        'confusion_matrix': cm,
        'confusion_matrix_norm': cm_norm
    })

# Summarize results
df_results = pd.DataFrame([{k:v for k,v in r.items() if k not in ('confusion_matrix','confusion_matrix_norm')} for r in results])
print('\nSummary table:')
df_results_sorted = df_results.sort_values('f1_macro', ascending=False).reset_index(drop=True)
df_results_sorted[['model','accuracy','balanced_accuracy','precision_macro','recall_macro','f1_macro','fit_time_s','pred_time_s']]

# Save metrics and df
df_results_sorted.to_csv(os.path.join(RESULTS_DIR, 'cifar10_results.csv'), index=False)

# Save metrics plot (seaborn grouped bar) and export vector
plt.figure(figsize=(10,5), dpi=200)
df_plot = df_results_sorted[['model','accuracy','balanced_accuracy','f1_macro']].melt(id_vars='model', var_name='metric', value_name='score')
sns.barplot(data=df_plot, x='model', y='score', hue='metric', palette='Set2')
plt.xticks(rotation=45, ha='right')
plt.ylim(0,1)
plt.ylabel('Score')
plt.title('Model comparison: Accuracy / Balanced / F1 (macro)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cifar10_evaluation_metrics.png'), dpi=300)
print('Saved cifar10_evaluation_metrics.png')
plt.close()

# Plot normalized confusion matrix of best model (by f1) with labels and export vector
best_name = df_results_sorted.loc[0, 'model']
best_r = next(r for r in results if r['model'] == best_name)
cm_best_norm = best_r['confusion_matrix_norm']
fig, ax = plt.subplots(figsize=(9,7), dpi=200)
sns.heatmap(cm_best_norm, annot=True, fmt='.2f', cmap='viridis', ax=ax)
ax.set_title(f"Normalized confusion matrix - {best_name}")
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_xticklabels(classes, rotation=45)
ax.set_yticklabels(classes, rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cifar10_best_confusion_norm.png'), dpi=300)
print('Saved cifar10_best_confusion_norm.png')
plt.close()

# Display results
df_results_sorted

Evaluating: KPCA + LDA (custom)
Evaluating: kNN (KPCA)
Evaluating: Logistic (KPCA)
Evaluating: RandomForest (KPCA)
Evaluating: PCA + LDA (custom)
Evaluating: kNN (PCA)
Evaluating: SVM (PCA)

Summary table:
Saved cifar10_evaluation_metrics.png
Saved cifar10_best_confusion_norm.png


,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,fit_time_s,pred_time_s
0,SVM (PCA),0.405,0.401363,0.403007,0.401363,0.399848,0.172455,0.118245
1,KPCA + LDA (custom),0.351,0.345883,0.349546,0.345883,0.346952,0.004511,0.004996
2,PCA + LDA (custom),0.334,0.328239,0.331015,0.328239,0.329167,0.003510,0.005005
3,RandomForest (KPCA),0.328,0.326785,0.331487,0.326785,0.322947,0.289451,0.024887
4,Logistic (KPCA),0.340,0.332993,0.355733,0.332993,0.316176,0.021072,0.000000
5,kNN (KPCA),0.271,0.273293,0.338558,0.273293,0.250542,0.001002,0.006015
6,kNN (PCA),0.267,0.269474,0.329572,0.269474,0.245067,0.001001,0.008103


In [12]:
# Compare custom KPCA+LDA with sklearn's KernelPCA + LDA
from sklearn.decomposition import KernelPCA as SKKernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as SKLDA
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score, confusion_matrix, classification_report

# Flatten images for sklearn (expects 2D input)
X_tr_flat = X_train_subset.reshape(X_train_subset.shape[0], -1)
X_te_flat = X_test_subset.reshape(X_test_subset.shape[0], -1)

# Fit sklearn KernelPCA (RBF) and transform
sk_kpca = SKKernelPCA(n_components=100, kernel='rbf', gamma=1e-5)
X_tr_sk = sk_kpca.fit_transform(X_tr_flat)
X_te_sk = sk_kpca.transform(X_te_flat)

# Fit sklearn LDA on sklearn-KPCA features
sk_lda = SKLDA(n_components=9)
sk_lda.fit(X_tr_sk, y_train_subset)
y_pred_sk = sk_lda.predict(X_te_sk)

# Metrics
acc_sk = accuracy_score(y_test_subset, y_pred_sk)
bal_sk = balanced_accuracy_score(y_test_subset, y_pred_sk)
p_sk, r_sk, f_sk, _ = precision_recall_fscore_support(y_test_subset, y_pred_sk, average='macro', zero_division=0)
print('sklearn KPCA+LDA Accuracy: {:.4f}'.format(acc_sk))
print(classification_report(y_test_subset, y_pred_sk, target_names=classes))

# Confusion matrix
cm_sk = confusion_matrix(y_test_subset, y_pred_sk)
row_sums = cm_sk.sum(axis=1)
cm_sk_norm = np.divide(cm_sk.astype(float), row_sums[:, None], out=np.zeros_like(cm_sk,dtype=float), where=row_sums[:, None]!=0)

# Numerical equivalence checks (allowing sign flips and column permutation)
# A = custom LDA test projection, B = sklearn LDA test projection (before alignment)
X_test_lda_custom = X_test_lda  # from earlier custom LDA transform
X_test_lda_sk_raw = sk_lda.transform(X_te_sk)
A = X_test_lda_custom.copy()
B = X_test_lda_sk_raw.copy()
# Correlation matrix between components (A.T vs B.T)
C = np.corrcoef(A.T, B.T)[:A.shape[1], A.shape[1]:]
# Find best matching sklearn column for each custom column (by absolute correlation)
perm = np.argmax(np.abs(C), axis=1)
B_perm = B[:, perm]
# Compute sign corrections per component
signs = np.array([np.sign(np.corrcoef(A[:,i], B_perm[:,i])[0,1]) if not np.allclose(B_perm[:,i],0) else 1.0 for i in range(A.shape[1])])
B_aligned = B_perm * signs[None, :]
# Report alignment quality
maxdiff = np.max(np.abs(A - B_aligned))
print(f'After permutation+sign alignment, max abs diff = {maxdiff:.6e}')
if np.allclose(A, B_aligned, atol=1e-6):
    print('Projections match up to permutation and sign (within 1e-6).')
else:
    print('Projections differ after alignment (max abs diff=', maxdiff, ').')

# Compare predictions (they should be identical regardless of axis signs)
try:
    same_preds = np.sum(y_pred == y_pred_sk)
    print(f'Number of identical predicted labels (custom vs sklearn): {same_preds} / {len(y_pred_sk)}')
except NameError:
    print('Custom predictions (y_pred) not available in this scope to compare.')

# For plotting, use the aligned sklearn projection so visuals match the custom projection
X_test_lda_sk = B_aligned
fig, axs = plt.subplots(1,2,figsize=(14,6), dpi=200)
sc0 = axs[0].scatter(X_test_lda_custom[:,0], X_test_lda_custom[:,1], c=y_test_subset, cmap='tab10', s=30, alpha=0.7)
axs[0].set_title('Custom KPCA+LDA (test projection)')
axs[0].set_xlabel('LD1'); axs[0].set_ylabel('LD2')
handles, _ = sc0.legend_elements()
axs[0].legend(handles, classes, bbox_to_anchor=(1.05,1), loc='upper left')
sc1 = axs[1].scatter(X_test_lda_sk[:,0], X_test_lda_sk[:,1], c=y_test_subset, cmap='tab10', s=30, alpha=0.7)
axs[1].set_title('sklearn KPCA + sklearn LDA (test projection - aligned)')
axs[1].set_xlabel('LD1'); axs[1].set_ylabel('LD2')
handles1, _ = sc1.legend_elements()
axs[1].legend(handles1, classes, bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'kpca_lda_comparison_scatter.png'), dpi=300)
print('Saved kpca_lda_comparison_scatter.png')
plt.close()

# Confusion heatmap for sklearn KPCA+LDA
fig, ax = plt.subplots(figsize=(9,7), dpi=200)
sns.heatmap(cm_sk, annot=True, fmt='d', cmap='viridis', ax=ax)
ax.set_title('Confusion matrix - sklearn KPCA+LDA')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_xticklabels(classes, rotation=45)
ax.set_yticklabels(classes, rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'kpca_lda_sklearn_confusion.png'), dpi=300)
print('Saved kpca_lda_sklearn_confusion.png')
plt.close()

# Append sklearn entry to results list for later inspection
try:
    results.append({
        'model': 'sklearn KPCA + LDA',
        'accuracy': acc_sk,
        'balanced_accuracy': bal_sk,
        'precision_macro': p_sk,
        'recall_macro': r_sk,
        'f1_macro': f_sk,
        'fit_time_s': None,
        'pred_time_s': None,
        'confusion_matrix': cm_sk,
        'confusion_matrix_norm': cm_sk_norm
    })
except NameError:
    pass



sklearn KPCA+LDA Accuracy: 0.3560
              precision    recall  f1-score   support

    airplane       0.52      0.45      0.48       106
  automobile       0.37      0.40      0.38       106
        bird       0.19      0.24      0.21        89
         cat       0.22      0.20      0.21        84
        deer       0.29      0.34      0.31        91
         dog       0.36      0.34      0.35       107
        frog       0.38      0.38      0.38       106
       horse       0.38      0.33      0.35        99
        ship       0.49      0.50      0.50       103
       truck       0.37      0.33      0.35       109

    accuracy                           0.36      1000
   macro avg       0.36      0.35      0.35      1000
weighted avg       0.36      0.36      0.36      1000

After permutation+sign alignment, max abs diff = 4.852664e+00
Projections differ after alignment (max abs diff= 4.852664231229861 ).
Number of identical predicted labels (custom vs sklearn): 979 / 1000
Saved

## Summary

"This notebook implemented KPCA + LDA for multi-class classification on CIFAR-10. The custom implementations were compared with standard PCA + LDA and other classifiers like kNN and SVM.",

Key points:
- KPCA provides nonlinear dimensionality reduction.
- LDA performs supervised classification in the reduced space.
- Results show the performance on subsets due to computational constraints.